# Claim Approval — Model Experiment Summary

This notebook summarizes the optimization experiment produced by `src/run_pipeline.py`.
It reads the artifacts in `reports/` and `models/` — **run the pipeline first** if they are missing:

```bash
python src/run_pipeline.py
```

**Goal:** predict `status` (Declined = positive class) on an imbalanced dataset, choosing the
model by **validation PR-AUC** rather than accuracy.

In [ ]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
REPORTS = ROOT / 'reports'
dash = json.loads((REPORTS / 'dashboard_data.json').read_text())
print('Dataset:', dash['overview']['dataset_name'])
print('Rows:', dash['overview']['n_rows'], '| Features:', dash['overview']['n_features'])
print('Class counts:', dash['overview']['class_counts'])
print('Imbalance ratio (Completed:Declined):', dash['overview']['imbalance_ratio'])

## 1. Model comparison (baseline vs optimized)

In [ ]:
comp = pd.read_csv(REPORTS / 'model_comparison.csv')
comp.style.background_gradient(subset=['val_pr_auc', 'test_pr_auc'], cmap='Greens') \
    if hasattr(comp, 'style') else comp

## 2. Best model & final recommendation

In [ ]:
best = dash['best_model']
print('Best model:', best['model'], f"({best['stage']})")
print('Imbalance strategy:', best['imbalance_strategy'])
print('Threshold:', best['threshold'])
print('\nTest metrics:')
for k in ['pr_auc', 'roc_auc', 'f1_declined', 'recall_declined', 'precision_declined', 'balanced_accuracy', 'accuracy']:
    print(f'  {k:20} {best["test_metrics"][k]:.4f}')
print('\nBest hyperparameters:')
print(json.dumps(best['params'], indent=2))
print('\n--- Final recommendation ---')
print(json.dumps(dash['final_recommendation'], indent=2))

## 3. Plain-English explanation

In [ ]:
from IPython.display import Markdown
Markdown(best['explanation'])

## 4. Key figures for the best model

In [ ]:
from IPython.display import Image, display
for key, fig in best['figures'].items():
    p = ROOT / fig
    if p.exists():
        print(key)
        display(Image(filename=str(p)))

## 5. Threshold analysis (validation)

In [ ]:
import matplotlib.pyplot as plt
ta = pd.DataFrame(dash['threshold_analysis'])
fig, ax = plt.subplots(figsize=(7, 4))
for col, c in [('precision_declined', 'C0'), ('recall_declined', 'C3'), ('f1_declined', 'C2')]:
    ax.plot(ta['threshold'], ta[col], label=col, color=c)
ax.axvline(best['threshold'], ls='--', color='k', label=f"selected = {best['threshold']}")
ax.set_xlabel('threshold'); ax.set_ylabel('score'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Threshold vs Precision / Recall / F1 (Declined)')
plt.show()

## 6. Optuna search summary
Best validation PR-AUC found per family, and the most influential hyperparameters.

In [ ]:
for model, info in dash['optuna'].items():
    print(f"{model:13} best val PR-AUC = {info['best_value']:.4f}  ({info['n_trials']} trials)")
    if info['param_importances']:
        top = sorted(info['param_importances'].items(), key=lambda kv: -kv[1])[:3]
        print('   top params:', ', '.join(f'{k}={v:.2f}' for k, v in top))